In [1]:
from platform import python_version
print(python_version())

3.11.14


In [2]:
import os, sys, yaml
from pathlib import Path
from dotenv import load_dotenv

import numpy as npmtd
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
pd.set_option("display.precision", 3)

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.MTD_lib import MTD
from libs.GDC_lib import GDC
from libs.calc_degs_lib import CALC_DEGS
# from libs.dashcyto_lib import DASH_CYTO
from libs.config_lib import Config

from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

with open('../params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

# print(dic_yml)

ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [3]:
email = os.getenv('email')

i_project=0

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

PROG_ID = 'TCGA'
PSI_ID = 'TCGA-BRCA'
PSI_ID = 'TCGA-ACC'
PSI_ID = 'TCGA-CESC'
PSI_ID = 'TCGA-PAAD'

ROOT0_DATA = ROOT0 / "data"
root_colab = ROOT0_DATA / 'colab'
root_project = ROOT0_DATA / PROG_ID

disease = PSI_ID

root_project = create_dir(ROOT0_DATA, s_project)
root_disease = create_dir(root_project, PSI_ID)

CONTEXT_DISESE = 'xxxx'
context_disease = CONTEXT_DISESE

gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

LFC_cut_inf = dic_yml['LFC_cut_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
ptw_min_num_of_degs_cut = dic_yml['ptw_min_num_of_degs_cut']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(root0=ROOT0, root_disease=root_disease, disease=disease, case_list=case_list)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1

LFC_cut, lfc_FDR_cut, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={LFC_cut:.3f}; fdr={lfc_FDR_cut:.3f} - LFC_cut_inf={LFC_cut_inf:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

Best parameter file for LFC does not exist /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD/config/all_lfc_cutoffs_TCGA-PAAD.tsv
project 'TCGA', s_project 'TCGA'
G/P LFC cutoffs: lfc=1.000; fdr=0.050 - LFC_cut_inf=0.400
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [4]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, 
          root0=ROOT0, root0_data=ROOT0_DATA, prog_id=PROG_ID, psi_id=PSI_ID,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, ptw_min_num_of_degs_cut=ptw_min_num_of_degs_cut,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          LFC_cut_inf=LFC_cut_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

print(">>> Roots", mtd.root0, mtd.root_disease)
case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, LFC_cut=1, lfc_FDR_cut=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=False, verbose=False)
# print("\nEcho Parameters:")
# print(mtd.echo_parameters())

>>> Roots /home/flavio/uv/perturb_agent /home/flavio/uv/perturb_agent/data/TCGA/TCGA-PAAD
>>> Tumor


### Get all programs

In [5]:
gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

#--------- chose a disease --------------------
DISEASE_ID = 'ACC'
DISEASE_ID = 'PAAD'

In [ ]:
verbose=False
force=False

imax_tumor=250
imax_normal=50

gdc = GDC(root0=ROOT0, root0_data=ROOT0_DATA, memory_restriction=False)

exclude_prog_list=['CCLE']

dfn_tumor, dfn_normal, df_gtex, df_summ = gdc.get_all_data_from_disease(disease_id=DISEASE_ID, 
                                                           imax_tumor=imax_tumor, imax_normal=imax_normal,
                                                           exclude_prog_list=exclude_prog_list,
                                                           force=force, verbose=verbose)
print("\n")
print(">> dfn_tumor", dfn_tumor.shape)
print(">> dfn_normal", dfn_normal.shape)

df_summ



>> dfn_tumor (60616, 460)
>> dfn_normal (60616, 103)


,prog_id,psi_id,disease_id,primary_site,n_tumors,n_normals
0,CPTAC,CPTAC-PAAD,PAAD,Pancreas,213,50
1,CPTAC,CPTAC-PAAD_GDC,PAAD,Pancreas,213,50
2,TCGA,TCGA-PAAD,PAAD,Pancreas,31,1


In [7]:
dfn_normal.head(3)

,geneid,symbol,biotype,1,2,3,4,5,6,7,...,91,92,93,94,95,96,97,98,99,100
0,ENSG00000000003,TSPAN6,protein_coding,1,1,0,1,0,2,1,...,0,0,1,1,1,1,0,2,0,0
1,ENSG00000000005,TNMD,protein_coding,7,10,14,3,7,14,5,...,11,13,3,9,16,12,17,9,7,9
2,ENSG00000000419,DPM1,protein_coding,6,3,2,2,0,6,8,...,11,4,6,9,7,5,15,5,1,9


### Batch effect correction and cpm normalization

In [8]:
force=False
verbose=False

perc_min_samples=0.25; top_n=10_000

df_sel, df_cpm, df_gene_annot = gdc.calc_expression_and_batch(dfn_tumor=dfn_tumor, group='Tumor', 
                                                           perc_min_samples=perc_min_samples, top_n=top_n,
                                                           force=force, verbose=verbose)

dfn, df_gene_annot = gdc.calc_cpm_merge_turmor_and_normal(dfn_tumor=dfn_tumor, dfn_normal=dfn_normal, 
                                                                   perc_min_samples=perc_min_samples, top_n=top_n,
                                                                   verbose=verbose)

dfall = gdc.dfall
print(dfn.shape, dfall.shape)
dfall.tail(3)

(10000, 557) (20024, 761)


,geneid,symbol,biotype,1,2,3,4,5,6,7,...,548,549,550,551,552,553,554,555,556,557
20021,ENSG00000288642,CDR1,protein_coding,6.854,2.566,9.519,3.949,8.535,5.713,4.761,...,7.463,7.407,6.449,6.805,6.270,7.023,4.828,8.240,7.913,6.957
20022,ENSG00000288663,AC073611.1,lncRNA,2.212,1.401,2.708,1.210,2.610,1.072,0.926,...,1.670,1.128,1.517,0.794,2.164,3.307,7.037,2.170,1.360,0.878
20023,ENSG00000288675,AP006621.6,protein_coding,3.305,4.532,3.454,3.410,2.236,3.973,3.186,...,1.670,1.913,1.693,2.434,2.017,2.630,1.758,2.644,2.511,2.488


In [9]:
dfn.tail(3)

,1,2,3,4,5,6,7,8,9,10,...,548,549,550,551,552,553,554,555,556,557
geneid,,,,,,,,,,,,,,,,,,,,,
ENSG00000271964,4.291,1.952,3.128,2.305,3.807,2.929,3.386,2.153,2.992,3.464,...,6.443,6.312,5.553,6.232,5.973,2.629,4.628,2.808,5.310,5.348
ENSG00000047410,4.150,2.838,4.896,3.630,6.694,2.334,4.919,4.360,3.539,1.914,...,6.935,5.994,7.534,6.989,6.148,3.833,6.105,4.143,7.761,6.174
ENSG00000145246,1.687,0.000,3.719,2.171,1.258,0.452,0.536,3.708,0.450,0.000,...,3.793,4.877,3.839,1.858,3.056,1.448,0.455,1.223,2.115,3.930


In [10]:
dfn.iloc[:5,450:470]

,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470
geneid,,,,,,,,,,,,,,,,,,,,
ENSG00000270641,0.350,0.339,9.473,5.700,0.359,8.507,0.339,14.700,5.580,3.549,2.690,4.611,14.728,2.912,14.708,4.117,14.095,4.408,14.690,3.393
ENSG00000262619,1.068,1.927,0.055,0.734,0.359,0.522,0.756,3.069,4.014,2.737,2.011,2.395,7.514,0.861,1.641,4.040,2.361,2.550,10.445,2.429
ENSG00000281383,0.242,0.885,0.301,0.000,0.249,0.000,0.398,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
ENSG00000189223,1.036,3.125,1.004,1.756,6.449,6.365,2.930,4.403,4.074,4.778,5.780,5.042,9.829,4.886,2.733,3.958,4.552,5.256,9.117,2.776
ENSG00000256462,0.000,0.000,0.000,0.000,0.066,0.000,0.000,0.000,0.000,0.000,0.683,0.000,0.765,0.000,0.000,0.933,1.126,0.000,0.759,11.727


### Batch effect correction

In [11]:
force=False
verbose=True

perc_min_samples=0.25; top_n=10_000

df_combat = gdc.calc_cpm_merge_turmor_and_normal_batch_correction(dfn_tumor=dfn_tumor, dfn_normal=dfn_normal,
                                                                  perc_min_samples=perc_min_samples, top_n=top_n,
                                                                  force=force, verbose=verbose)
print(df_combat.shape)
df_combat.head(3)

Table opened ((10000, 458)) at '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/Tumor_expression_sel_log_CPM_top_10000_genes_all_samples.tsv'
Table opened ((20024, 458)) at '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/Tumor_expression_log_CPM_all_samples.tsv'
Table opened ((10000, 3)) at '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/Tumor_expression_gene_annot_top_10000_genes_all_samples.tsv'
Table opened ((20024, 1263)) at '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/cpm_log_all_genes_all_rows.tsv'
Table opened ((10000, 560)) at '/home/flavio/uv/perturb_agent/data/multi_progs/PAAD/lfc/combat_log_exp_tumor_and_normal.tsv'
(10000, 560)


,geneid,symbol,biotype,1,2,3,4,5,6,7,...,548,549,550,551,552,553,554,555,556,557
0,ENSG00000270641,TSIX,lncRNA,13.955,13.211,13.898,13.478,13.710,11.679,13.350,...,2.868,3.028,5.113,2.322,3.620,4.659,13.198,4.548,3.652,14.046
1,ENSG00000262619,LINC00621,lncRNA,10.371,10.430,12.476,12.408,14.294,9.301,12.063,...,1.102,1.409,2.019,0.461,0.589,2.182,2.404,4.046,3.690,2.335
2,ENSG00000281383,FP671120.7,lncRNA,3.266,-0.078,10.497,-0.078,11.041,5.579,2.968,...,-0.146,-0.146,-0.146,-0.146,-0.146,-0.146,1.137,-0.146,0.574,-0.146


### Metadata

In [12]:
print(pd.crosstab(gdc.df_metadata["dataset"], gdc.df_metadata["condition"]))

condition       normal  tumor
dataset                      
CPTAC-PAAD          50    213
CPTAC-PAAD_GDC      50    213
TCGA-PAAD            0     31


### Loop all clusters

In [13]:
disease_id=DISEASE_ID
imax_tumor=250
imax_normal=50
exclude_prog_list=['CCLE']
n_components = 10
n_umap_neighbors=5; min_umap_dist=0.2; umap_metric="euclidean"
method_hca="ward"; hca_criterion="maxclust"
LFC_cutoff=1; FDR_cutoff=0.05

force=False
verbose=False

group = 'Tumor'
min_clusters = 3

for n_clusters in range(3, 10+1):

    print(f"Clusters: {n_clusters}")

    fname = f'all_cluster_degs_for_{n_clusters}_clusterization.txt'
    filename = gdc.root_mprog_lfc / fname

    if filename.exists() and not force:
        print(f"\tFile {fname} already exists. Skipping.")
        continue

    # for silhouette
    max_clusters = n_clusters+2

    df_cluster, df_pca, df_umap = gdc.cluster_PCA_HCA_UMAP(df_combat, group=group, n_clusters=n_clusters, 
                                                    n_components=n_components, min_clusters=min_clusters, max_clusters=max_clusters,
                                                    n_umap_neighbors=n_umap_neighbors, min_umap_dist=min_umap_dist, umap_metric=umap_metric,
                                                    method_hca=method_hca, hca_criterion=hca_criterion,
                                                    LFC_cutoff=LFC_cutoff, FDR_cutoff=FDR_cutoff,
                                                    force=force, verbose=verbose)
    
    df_hca = gdc.df_hca
    df_metadata = gdc.df_metadata
    
    lista = df_metadata[df_metadata.condition == 'normal'].index.to_list()
    mini = np.min(lista)

    normal_clusters = np.unique(df_hca[df_hca['sample'] >= mini].cluster)

    max_cluster = df_hca.cluster.max()
    all_clusters_text = ''

    for nclu in range(1, max_cluster + 1):

        print(f"\tProcessing cluster {nclu}/{n_clusters}")

        df_lfc, df_lfc_ori, degs_txt, degs_first2000, degs_for_AI_analysis, msg = \
            gdc.calc_limma_inmoose(prog_id=PROG_ID, psi_id=PSI_ID, ncluster=nclu,
                            lfc_cutoff = LFC_cutoff, fdr_cutoff = FDR_cutoff,
                            force = force, verbose = verbose )
        
        text = f"\tCluster {nclu}:\n{degs_for_AI_analysis}"

        if all_clusters_text == '':
            all_clusters_text = text
        else:
            all_clusters_text += "\n\n----------------------------------\n" + text
        print(msg, '\n----------------------------------')

    print(f"------------------- end clusterization {n_clusters}---------------")
    write_txt(all_clusters_text, fname, gdc.root_mprog_lfc)

print(f"\n------------------- end -----------------------")

Clusters: 3
	File all_cluster_degs_for_3_clusterization.txt already exists. Skipping.
Clusters: 4
	File all_cluster_degs_for_4_clusterization.txt already exists. Skipping.
Clusters: 5
	File all_cluster_degs_for_5_clusterization.txt already exists. Skipping.
Clusters: 6
	File all_cluster_degs_for_6_clusterization.txt already exists. Skipping.
Clusters: 7
	File all_cluster_degs_for_7_clusterization.txt already exists. Skipping.
Clusters: 8
	File all_cluster_degs_for_8_clusterization.txt already exists. Skipping.
Clusters: 9
	File all_cluster_degs_for_9_clusterization.txt already exists. Skipping.
Clusters: 10
	File all_cluster_degs_for_10_clusterization.txt already exists. Skipping.

------------------- end -----------------------


In [ ]:
gdc.disease_id